# LegalMitra: LoRA Adapter Merge & 4-bit GGUF Export

**Objective:**
1. Merge the trained 167 MB LegalMitra LoRA adapter into the base `unsloth/Meta-Llama-3.1-8B-Instruct`.
2. Convert the merged FP16 model into **GGUF format** using `llama.cpp` (built with CMake).
3. Quantize the GGUF model into `q4_k_m` (4-bit medium quantization, ~4.6 GB).
4. Output a standalone, offline model file `legalmitra-8b-q4_k_m.gguf` for local CPU inference with zero GPU required.

**Hardware Setting:** Kaggle Free GPU (T4 x2 or P100) | **Internet:** Enabled

## Step 1: Install Compatible Transformers & Build llama.cpp (Self-Healing Session Restart)

In [ ]:
# Upgrade transformers, accelerate, bitsandbytes, and peft together to match the exact training environment
!pip install -q -U transformers accelerate bitsandbytes peft datasets sentencepiece huggingface_hub gguf

# Clone llama.cpp and compile using CMake
!rm -rf /kaggle/working/llama.cpp
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /kaggle/working/llama.cpp
!cmake -B /kaggle/working/llama.cpp/build -S /kaggle/working/llama.cpp -DGGML_CUDA=ON
!cmake --build /kaggle/working/llama.cpp/build --config Release -j4 --target llama-quantize llama-cli
!pip install -q -r /kaggle/working/llama.cpp/requirements.txt

print("Dependencies updated. Now restarting python kernel so new modules load cleanly...")
import os
os._exit(00)

## Step 2: Locate the Uploaded LoRA Adapter Dataset

In [ ]:
import os
import glob

# Search Kaggle input directories for the adapter
adapter_candidates = glob.glob('/kaggle/input/**/adapter_model.safetensors', recursive=True)

if adapter_candidates:
    adapter_dir = os.path.dirname(adapter_candidates[0])
    print(f"Found LegalMitra LoRA adapter at: {adapter_dir}")
else:
    adapter_dir = '/kaggle/working/legalmitra_llama3_lora_final'
    if not os.path.exists(adapter_dir):
        raise FileNotFoundError(
            "LoRA adapter not found! Please attach your 'legalmitra-lora-adapter' Kaggle dataset "
            "via '+ Add Input' on the right panel."
        )
    print(f"Using local session adapter at: {adapter_dir}")

print("Adapter files:", os.listdir(adapter_dir))

## Step 3: Merge LoRA Adapter with Base Model (FP16)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_name = "unsloth/Meta-Llama-3.1-8B-Instruct"
merged_output_dir = "/kaggle/working/legalmitra_merged_fp16"

print(f"Loading base model in FP16 from: {base_model_name}...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print(f"Applying LegalMitra LoRA adapter from: {adapter_dir}...")
model = PeftModel.from_pretrained(base_model, adapter_dir)

print("Merging LoRA weights directly into the base weights...")
merged_model = model.merge_and_unload()

print(f"Saving merged FP16 model to: {merged_output_dir}...")
merged_model.save_pretrained(merged_output_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_output_dir)
print("SUCCESS: Model successfully merged into full FP16 weights.")

## Step 4: Convert Merged Model to GGUF & Quantize to 4-bit (q4_k_m)

In [ ]:
import os

fp16_gguf_path = "/kaggle/working/legalmitra-8b-fp16.gguf"
q4_gguf_path = "/kaggle/working/legalmitra-8b-q4_k_m.gguf"

print("Step 4A: Converting merged weights to FP16 GGUF...")
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py \
    /kaggle/working/legalmitra_merged_fp16 \
    --outfile {fp16_gguf_path} \
    --outtype f16

print("Step 4B: Quantizing FP16 GGUF to q4_k_m (4-bit Medium)...")
quantize_bin = "/kaggle/working/llama.cpp/build/bin/llama-quantize"
if not os.path.exists(quantize_bin):
    quantize_bin = "/kaggle/working/llama.cpp/llama-quantize"

!{quantize_bin} {fp16_gguf_path} {q4_gguf_path} q4_k_m

# Clean up large intermediate files to free disk space
if os.path.exists(fp16_gguf_path):
    os.remove(fp16_gguf_path)
!rm -rf /kaggle/working/legalmitra_merged_fp16

q4_size_gb = os.path.getsize(q4_gguf_path) / (1024 ** 3)
print(f"\n====================================================================")
print(f"SUCCESS: Standalone GGUF model created: {q4_gguf_path}")
print(f"Final 4-bit Quantized Size: {q4_size_gb:.2f} GB")
print(f"====================================================================")
print("You can now download 'legalmitra-8b-q4_k_m.gguf' from the Output tab on the right panel!")

## Step 5: Quick Inference Test (Sanity Check)

In [ ]:
test_prompt = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nWhat happens in case of minor theft under the Bharatiya Nyaya Sanhita, 2023?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

cli_bin = "/kaggle/working/llama.cpp/build/bin/llama-cli"
if not os.path.exists(cli_bin):
    cli_bin = "/kaggle/working/llama.cpp/llama-cli"

!{cli_bin} \
    -m /kaggle/working/legalmitra-8b-q4_k_m.gguf \
    -p "{test_prompt}" \
    -n 128 \
    --temp 0.2 \
    -ngl 33